# 💡 SQL Track – Solutions

This notebook contains the solutions of all **🏋️ Your turn** exercises of the SQL Track, in the order of the labs.

> **❗ Important:** Try each exercise yourself first – you learn far more from a query you struggled with than from one you copied. Come here to check your result or when you are stuck.

Every solution is a SQL cell you can run right here. Attach the notebook to **Serverless** compute or a **SQL warehouse** and set the default catalog first (replace it if you used a different catalog in Lab 0):

In [ ]:
USE CATALOG sunny_bay_germany;

> **📝 Note:** If you used a **prefix** in Lab 0, all table names start with it (e.g. `silver.jd_customer` instead of `silver.customer`).

## SQL 1 – Filtering, Aggregating and Joining Data

### SQL 1 › Step 2 › Task 4: Filter Rows with WHERE

Find all orders in `silver.sales_order` from the **Berlin café** (`store_id = 2`) that were paid with **Girocard** during the **last 7 days**. Show the order id, timestamp and payment method, newest first.

In [ ]:
SELECT order_id, order_ts, payment_method
FROM silver.sales_order
WHERE store_id = 2
  AND payment_method = 'Girocard'
  AND order_date >= current_date() - INTERVAL 7 DAYS
ORDER BY order_ts DESC;

### SQL 1 › Step 3 › Task 4: Summarize with GROUP BY and HAVING

The CRM extract `bronze.crm_customers` should contain one row per customer – but does it? Find all `customer_id`s that appear **more than once** and show how many rows each of them has.

In [ ]:
SELECT customer_id, COUNT(*) AS versions
FROM bronze.crm_customers
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY versions DESC, customer_id;

The CRM delivers **history rows**: when a customer changes their e-mail address or moves, a new version with a newer `updated_at` is added. You will learn how to keep only the latest version with `QUALIFY` in SQL 3 – and in Step 6 you will see what happens if you forget it.

### SQL 1 › Step 5 › Task 4: INNER JOIN – Only Matching Rows

Calculate the **gross revenue per product name** (`quantity * unit_price_eur - discount_eur`) and show the 10 best-selling products.

In [ ]:
SELECT
    p.product_name,
    ROUND(SUM(l.quantity * l.unit_price_eur - l.discount_eur), 2) AS gross_revenue_eur
FROM silver.sales_order_line AS l
INNER JOIN silver.product AS p
    ON l.product_id = p.product_id
GROUP BY p.product_name
ORDER BY gross_revenue_eur DESC
LIMIT 10;

### SQL 1 › Step 6 › Task 3: LEFT JOIN – Keep Everything from the Left

How many customers in `silver.customer` have **never had** a loyalty card? Use a `LEFT JOIN` and look for the `NULL`s.

In [ ]:
SELECT COUNT(*) AS customers_without_card
FROM silver.customer AS c
LEFT JOIN silver.loyalty_card AS lc
    ON c.customer_id = lc.customer_id
WHERE lc.card_no IS NULL;

This "LEFT JOIN + IS NULL" pattern is the classic way to find rows **without** a match. In Step 9 you will meet a shorter way to write it: the anti join.

### SQL 1 › Step 7 › Task 3: FULL OUTER JOIN – Compare Two Systems

Why do we compare `LOWER(TRIM(crm.email))` instead of simply `crm.email`? Count the matching rows (`INNER JOIN`) once **with** and once **without** `LOWER(TRIM(...))`. Then look at some CRM e-mails that are not already lower case.

In [ ]:
SELECT
    (SELECT COUNT(*)
     FROM bronze.crm_customers AS crm
     INNER JOIN bronze.web_accounts AS web ON crm.email = web.email)              AS raw_matches,
    (SELECT COUNT(*)
     FROM bronze.crm_customers AS crm
     INNER JOIN bronze.web_accounts AS web ON LOWER(TRIM(crm.email)) = web.email) AS normalized_matches;

SELECT customer_id, email
FROM bronze.crm_customers
WHERE email <> LOWER(TRIM(email))
LIMIT 20;

The CRM stores e-mails exactly as typed by the café staff – with capital letters and leading or trailing blanks. The web shop stores them in lower case. **Join keys must be normalized** before they can match. The silver table `silver.customer` does exactly this (and also removes the CRM history rows) before it combines both systems with a full outer join.

### SQL 1 › Step 8 › Task 4: LEFT SEMI JOIN – "Has at Least One Match"

Which loyalty cards were used for at least one café purchase during the **last 30 days**? Show card number, customer and tier.

In [ ]:
SELECT lc.card_no, lc.customer_id, lc.loyalty_tier
FROM silver.loyalty_card AS lc
LEFT SEMI JOIN silver.sales_order AS o
    ON  lc.card_no = o.loyalty_card_no
    AND o.order_date >= current_date() - INTERVAL 30 DAYS
ORDER BY lc.card_no;

### SQL 1 › Step 9 › Task 3: LEFT ANTI JOIN – "Has No Match"

Which products in the product master have **never been sold**? Write the query once with `LEFT ANTI JOIN` and once with `NOT EXISTS`. Then compare your result with the `has_sales` column of `gold.dim_product`.

In [ ]:
SELECT p.product_id, p.product_name, p.product_status, p.launched_on
FROM silver.product AS p
LEFT ANTI JOIN silver.sales_order_line AS l
    ON p.product_id = l.product_id;

SELECT p.product_id, p.product_name, p.product_status, p.launched_on
FROM silver.product AS p
WHERE NOT EXISTS (
    SELECT 1 FROM silver.sales_order_line AS l WHERE l.product_id = p.product_id
);

SELECT product_key, product_name, has_sales
FROM gold.dim_product
WHERE NOT has_sales;

You should find three products: one was discontinued before the sales window and two will be launched soon.

### SQL 1 › Step 10 › Task 3: CROSS JOIN – Every Combination

How many rows does the grid have **before** the anti join? Count the stores, the days of the sales window and the rows of the cross join, and check that rows = stores × days.

In [ ]:
SELECT
    COUNT(DISTINCT s.store_id) AS stores,
    COUNT(DISTINCT d.date)     AS days,
    COUNT(*)                   AS grid_rows
FROM silver.store AS s
CROSS JOIN gold.dim_date AS d
WHERE d.date BETWEEN (SELECT MIN(order_date) FROM silver.sales_order)
                 AND (SELECT MAX(order_date) FROM silver.sales_order);

## SQL 2 – Combining and Analyzing Data

### SQL 2 › Step 1 › Task 3: Stack Result Sets with UNION ALL and UNION

Build one list of all **cities** in which Sunny Bay has customers – from the CRM (`bronze.crm_customers.city`) and from the web shop (`bronze.web_accounts.city`). Every city should appear only once, sorted alphabetically.

In [ ]:
SELECT city FROM bronze.crm_customers
UNION
SELECT city FROM bronze.web_accounts
ORDER BY city;

`UNION` (without `ALL`) removes the duplicates. Note that `ORDER BY` at the end sorts the **combined** result.

### SQL 2 › Step 2 › Task 4: Compare Lists with MINUS and INTERSECT

Which e-mail addresses exist in **both** the CRM (`bronze.crm_customers`) and the web shop (`bronze.web_accounts`)? Remember the lesson from SQL 1: normalize the CRM e-mails with `LOWER(TRIM(...))`. Then count them.

In [ ]:
SELECT COUNT(*) AS emails_in_both_systems
FROM (
    SELECT LOWER(TRIM(email)) AS email FROM bronze.crm_customers
    INTERSECT
    SELECT email FROM bronze.web_accounts
) AS both_systems;

These are the **omnichannel** customers who buy in the cafés *and* online. `silver.customer` merges each of them into a single customer.

### SQL 2 › Step 3 › Task 3: Structure Queries with WITH … AS

The nested query in the lab finds the stores whose number of **Saturday** orders is above the average of all stores. It works, but it is hard to read. Rewrite it with two CTEs: `saturday_orders` (orders per store) and `average` (the average of these numbers).

In [ ]:
WITH saturday_orders AS (
    SELECT s.store_name, COUNT(*) AS saturday_orders
    FROM silver.sales_order AS o
    INNER JOIN silver.store AS s ON o.store_id = s.store_id
    WHERE DAYOFWEEK(o.order_date) = 7      -- 1 = Sunday ... 7 = Saturday
    GROUP BY s.store_name
),
average AS (
    SELECT AVG(saturday_orders) AS avg_saturday_orders
    FROM saturday_orders
)
SELECT so.store_name, so.saturday_orders
FROM saturday_orders AS so
CROSS JOIN average AS a
WHERE so.saturday_orders > a.avg_saturday_orders
ORDER BY so.saturday_orders DESC;

The CTE `saturday_orders` is used twice – once for the average and once for the final result. The `CROSS JOIN` with a one-row CTE is a common trick to attach a single value to every row.

### SQL 2 › Step 4 › Task 5: Analyze with Window Functions and PARTITION BY

For every loyalty card, find the **first café order** (the order with the earliest `order_ts`). Number the orders per card with `ROW_NUMBER()` in a CTE and keep only number 1. Show card number, order id and timestamp.

In [ ]:
WITH numbered_orders AS (
    SELECT
        loyalty_card_no,
        order_id,
        order_ts,
        ROW_NUMBER() OVER (PARTITION BY loyalty_card_no ORDER BY order_ts) AS order_no
    FROM silver.sales_order
    WHERE loyalty_card_no IS NOT NULL
)
SELECT loyalty_card_no, order_id, order_ts
FROM numbered_orders
WHERE order_no = 1
ORDER BY order_ts;

You cannot write `WHERE ROW_NUMBER() OVER (…) = 1` directly, because `WHERE` is evaluated **before** window functions. That's why we need the CTE – or the `QUALIFY` clause you will learn in SQL 3.

### SQL 2 › Step 5 › Task 4: Subtotals with ROLLUP, CUBE and GROUPING()

Show the gross revenue per **store** and **product category** (from `gold.dim_store` and `gold.dim_product`), with a subtotal per store and a grand total. Label the subtotal rows.

In [ ]:
SELECT
    s.store_name,
    p.product_category,
    SUM(f.gross_revenue_eur) AS revenue_eur,
    CASE
        WHEN GROUPING(s.store_name) = 1       THEN 'Grand total'
        WHEN GROUPING(p.product_category) = 1 THEN 'Store total'
        ELSE 'Category'
    END AS row_type
FROM gold.fact_coffee_sales AS f
INNER JOIN gold.dim_store   AS s ON f.store_key = s.store_key
INNER JOIN gold.dim_product AS p ON f.product_key = p.product_key
GROUP BY ROLLUP (s.store_name, p.product_category)
ORDER BY s.store_name NULLS LAST, p.product_category NULLS LAST;

### SQL 2 › Step 6 › Task 3: Walk Through Hierarchies with WITH RECURSIVE

Build the **category path** of every product, e.g. `All Products > Drink > Milk Coffee > Cappuccino`. Start the recursion at the root category (`parent_category_id IS NULL`) of `silver.product_category`, walk down the tree, and finally join the products to their category (`silver.product.category_id`).

In [ ]:
WITH RECURSIVE category_path (category_id, path, depth) AS (
    SELECT category_id, category_name, 1
    FROM silver.product_category
    WHERE parent_category_id IS NULL

    UNION ALL

    SELECT c.category_id, CONCAT(cp.path, ' > ', c.category_name), cp.depth + 1
    FROM silver.product_category AS c
    INNER JOIN category_path AS cp
        ON c.parent_category_id = cp.category_id
)
SELECT
    p.product_name,
    CONCAT(cp.path, ' > ', p.product_name) AS full_path,
    cp.depth + 1                           AS depth
FROM silver.product AS p
INNER JOIN category_path AS cp
    ON p.category_id = cp.category_id
ORDER BY full_path;

> **✨ Tip:** Have a look at `bundle/src/transformations/gold/dim_product.sql`: the gold layer flattens the category tree **without** recursion – it joins `product_category` twice (a *self-join*). That works because the tree has a fixed depth of three levels. Use recursion when the depth can vary (like an org chart).

## SQL 3 – Functions for the Gold Layer

### SQL 3 › Step 1 › Task 4: Flatten Arrays with EXPLODE

How many **units per SKU** were ordered online? Work directly on the raw JSON in `bronze.web_orders` (ignore cancelled orders) and sort by units.

In [ ]:
WITH web_items AS (
    SELECT EXPLODE(items) AS item
    FROM bronze.web_orders
    WHERE status <> 'cancelled'
)
SELECT item.sku, SUM(item.qty) AS units
FROM web_items
GROUP BY item.sku
ORDER BY units DESC;

### SQL 3 › Step 2 › Task 4: Keep the Right Row per Group with QUALIFY

In SQL 2 you ranked the products per store with a CTE. Now show only the **top 3 products per store** by gross revenue – in a single query with `QUALIFY` (use `gold.fact_coffee_sales`, `gold.dim_store` and `gold.dim_product`).

In [ ]:
SELECT
    s.store_name,
    p.product_name,
    SUM(f.gross_revenue_eur) AS revenue_eur
FROM gold.fact_coffee_sales AS f
INNER JOIN gold.dim_store   AS s ON f.store_key = s.store_key
INNER JOIN gold.dim_product AS p ON f.product_key = p.product_key
GROUP BY s.store_name, p.product_name
QUALIFY RANK() OVER (PARTITION BY s.store_name ORDER BY SUM(f.gross_revenue_eur) DESC) <= 3
ORDER BY store_name, revenue_eur DESC;

`QUALIFY` is evaluated **after** `GROUP BY`, so the window function can even rank aggregated values.

### SQL 3 › Step 3 › Task 4: Replace Missing Values with COALESCE

Online orders have a delivery city (`ship_to_city`), café orders don't. Count the orders per **"order city"**: the delivery city for online orders, otherwise the city of the store (join `silver.store`).

In [ ]:
SELECT
    COALESCE(o.ship_to_city, s.city) AS order_city,
    COUNT(*)                         AS orders
FROM silver.sales_order AS o
INNER JOIN silver.store AS s
    ON o.store_id = s.store_id
GROUP BY COALESCE(o.ship_to_city, s.city)
ORDER BY orders DESC;

### SQL 3 › Step 4 › Task 3: Derive Values with CASE and IFF

Count the customers in `silver.customer` per **age group**: `Under 30`, `30–49`, `50 and older` and `Unknown` (web-shop-only customers have no birth date). Calculate the age as `YEAR(current_date()) - YEAR(birth_date)`.

In [ ]:
SELECT
    CASE
        WHEN birth_date IS NULL                              THEN 'Unknown'
        WHEN YEAR(current_date()) - YEAR(birth_date) < 30    THEN 'Under 30'
        WHEN YEAR(current_date()) - YEAR(birth_date) < 50    THEN '30–49'
        ELSE '50 and older'
    END      AS age_group,
    COUNT(*) AS customers
FROM silver.customer
GROUP BY 1
ORDER BY 1;

Note that the `NULL` check comes **first**: a comparison with `NULL` is never true, so without it those customers would land in the `ELSE` branch.

### SQL 3 › Step 5 › Task 4: Translate Codes with DECODE

Make the raw ERP product list in `bronze.erp_products` readable: decode `channel_code` (`S` = Café only, `O` = Online only, `B` = Café and online), `status_code` (`A` = Active, `D` = Discontinued) and `tax_class` (`S` = 0.19, `R` = 0.07 VAT rate).

In [ ]:
SELECT
    sku,
    product_name,
    DECODE(channel_code, 'S', 'Café only', 'O', 'Online only', 'B', 'Café and online', 'Unknown') AS sales_channels,
    DECODE(status_code, 'A', 'Active', 'D', 'Discontinued', 'Unknown')                            AS product_status,
    DECODE(tax_class, 'S', 0.19, 'R', 0.07)                                                       AS vat_rate
FROM bronze.erp_products
ORDER BY sku;

Compare your result with `silver.product`, which decodes the same columns.

### SQL 3 › Step 6 › Task 4: Pick the Value at the Maximum with MAX_BY

Find the **best day** of every store: the date with the highest gross revenue and the revenue of that day.

In [ ]:
WITH daily AS (
    SELECT s.store_name, d.date, SUM(f.gross_revenue_eur) AS revenue_eur
    FROM gold.fact_coffee_sales AS f
    INNER JOIN gold.dim_store AS s ON f.store_key = s.store_key
    INNER JOIN gold.dim_date  AS d ON f.date_key = d.date_key
    GROUP BY s.store_name, d.date
)
SELECT
    store_name,
    MAX_BY(date, revenue_eur) AS best_day,
    MAX(revenue_eur)          AS best_day_revenue_eur
FROM daily
GROUP BY store_name
ORDER BY store_name;

### SQL 3 › Step 7 › Task 3: Conditional Aggregates with FILTER (WHERE …)

For every **café**, count the orders paid with **cash** and the orders paid **cashless** (all other payment methods), and calculate the cash share in percent.

In [ ]:
SELECT
    s.store_name,
    COUNT(*) FILTER (WHERE o.payment_method = 'Cash')                       AS cash_orders,
    COUNT(*) FILTER (WHERE o.payment_method <> 'Cash')                      AS cashless_orders,
    ROUND(100 * COUNT(*) FILTER (WHERE o.payment_method = 'Cash') / COUNT(*), 1) AS cash_share_pct
FROM silver.sales_order AS o
INNER JOIN silver.store AS s
    ON o.store_id = s.store_id
WHERE o.sales_channel = 'In-store'
GROUP BY s.store_name
ORDER BY cash_share_pct DESC;

### SQL 3 › Step 8 › Task 4: Calculate with Dates – DATEADD and DATEDIFF

New loyalty members get a welcome bonus during their **first year** of membership. For every **active** card, calculate the date on which the first year ends and show whether the member is still within the first year today.

In [ ]:
SELECT
    card_no,
    customer_id,
    enrolled_on,
    CAST(DATEADD(YEAR, 1, enrolled_on) AS DATE)               AS first_year_ends_on,
    current_date() < CAST(DATEADD(YEAR, 1, enrolled_on) AS DATE) AS in_first_year
FROM silver.loyalty_card
WHERE is_active
ORDER BY enrolled_on DESC;

## 🎉 Done

Back to the labs: **SQL 1 – Filtering, Aggregating and Joining**, **SQL 2 – Combining and Analyzing Data** and **SQL 3 – Functions for the Gold Layer**.